# Lesson 08: Evaluation

**Goal:** compute retrieval metrics yourself and put an honest confidence interval on a score.

**Before you start**
- Theory: [lesson 08](../../docs/lessons/08_eval.md)
- Python concepts: comprehensions, `zip`, `random.Random`, `statistics`
- Files used: `src/docrag/eval/metrics.py` (no PDFs needed). Offline, no API key, no cost.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
import math
import random
import statistics

from docrag.eval.metrics import mrr_at_k, recall_at_k

# Three questions: retrieved pages in rank order, and the gold evidence pages.
runs = [([3, 1, 2], {1}), ([5, 4, 9], {4, 7}), ([2, 8], {6})]

## Step 1: Recall@k and MRR by hand

- **recall@k**: the share of gold pages that appear in the top k (`len(set(pages[:k]) & gold) / len(gold)`).
- **MRR@k**: 1 / rank of the *first* gold page in the top k, 0 if none (ranks start at 1).

Write both, then they're compared with the project's implementation for every question.

In [ ]:
def my_recall(pages: list[int], gold: set[int], k: int) -> float:
    return 0.0  # TODO


def my_mrr(pages: list[int], gold: set[int], k: int = 10) -> float:
    return 0.0  # TODO (enumerate(pages[:k], start=1))

In [ ]:
for pages, gold in runs:
    for k in (1, 2, 3):
        assert math.isclose(my_recall(pages, gold, k), recall_at_k(pages, gold, k)), (
            f"recall@{k} differs for {pages}, {gold}"
        )
    assert math.isclose(my_mrr(pages, gold), mrr_at_k(pages, gold)), (
        f"MRR differs for {pages}, {gold}"
    )
print("mean recall@3:", statistics.fmean(my_recall(p, g, 3) for p, g in runs))
print("mean MRR:     ", statistics.fmean(my_mrr(p, g) for p, g in runs))
print("OK")

## Step 2: A bootstrap confidence interval, per document

Questions about the same PDF are correlated: if a document parses badly, *all* its questions fail together. So resample **documents** (with replacement), not questions: pick `len(docs)` documents at random, pool their questions, take the mean, repeat 2,000 times, and read the 2.5% and 97.5% percentiles. `rng.choices(seq, k=n)` samples with replacement.

In [ ]:
# Synthetic but realistic: correctness depends strongly on the document.
gen = random.Random(42)
by_doc = {
    f"doc{d}": [int(gen.random() < p) for _ in range(8)]
    for d, p in enumerate([0.95, 0.9, 0.85, 0.2, 0.1, 0.8, 0.15, 0.9, 0.05, 0.7])
}
all_values = [v for vs in by_doc.values() for v in vs]
print(
    "mean correct:",
    statistics.fmean(all_values),
    "over",
    len(all_values),
    "questions,",
    len(by_doc),
    "docs",
)

In [ ]:
def doc_bootstrap_ci(
    by_doc: dict[str, list[int]], n: int = 2000, seed: int = 0
) -> tuple[float, float]:
    rng = random.Random(seed)
    docs = list(by_doc)
    means = []
    # TODO: n times: sample len(docs) documents with rng.choices, pool their values,
    #       append statistics.fmean(pooled) to means. Then sort and return the
    #       values at index int(0.025 * n) and int(0.975 * n).
    return 0.0, 0.0

In [ ]:
lo, hi = doc_bootstrap_ci(by_doc)
mean = statistics.fmean(all_values)
assert lo < mean < hi, f"The mean {mean:.3f} should lie inside [{lo:.3f}, {hi:.3f}]"

# For comparison: the (wrong) per-question bootstrap pretends questions are independent.
rng = random.Random(0)
q_means = sorted(
    statistics.fmean(rng.choices(all_values, k=len(all_values))) for _ in range(2000)
)
q_lo, q_hi = q_means[50], q_means[1950]
assert (hi - lo) > (q_hi - q_lo), (
    "Per-document intervals are wider when documents differ"
)
print(f"per document: [{lo:.3f}, {hi:.3f}]  width {hi - lo:.3f}")
print(
    f"per question: [{q_lo:.3f}, {q_hi:.3f}]  width {q_hi - q_lo:.3f}  (too optimistic)"
)
print("OK")

## Observation question

Why resample per document and not per question?

<details>
<summary>Answer</summary>

The questions aren't independent: the same PDF, the same parse, the same chunking decide many of them together. A per-question bootstrap treats 80 questions as 80 independent draws and gives an interval that is too narrow, so a 3-point difference between two configs looks significant when it isn't. Resampling documents keeps each document's questions together (a "cluster bootstrap") and gives the honest, wider interval. For comparing two configs on the *same* questions, a paired test is even better: the abstention experiment used McNemar's test on the questions where the two variants disagree.
</details>

## The real code

- Metrics: [`src/docrag/eval/metrics.py`](../../src/docrag/eval/metrics.py)
- Runner: [`src/docrag/eval/runner.py`](../../src/docrag/eval/runner.py)
- Judge: [`src/docrag/eval/judge.py`](../../src/docrag/eval/judge.py)
- Lesson: [`docs/lessons/08_eval.md`](../../docs/lessons/08_eval.md)